In [1]:
import pandas as pd
import re
from langdetect import detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

In [2]:
df = pd.read_csv('../cleanData/parsed/arguments_parsed.csv', encoding='utf-8')
print(df.shape)
print(df['debate'].nunique())
print(df.isna().sum())
df.head()

(97102, 6)
238
id           0
parent     222
type         0
debate       0
content      0
title        0
dtype: int64


,id,parent,type,debate,content,title
0,1,NaN,Thesis,according-to-the-bible-salvation-is-by-faith-a...,"According to the Bible, salvation is by faith ...","According to the Bible, salvation is by faith ..."
1,1.1,1,Con,according-to-the-bible-salvation-is-by-faith-a...,Baptism is also required for salvation.,"According to the Bible, salvation is by faith ..."
2,1.1.1,1.1,Pro,according-to-the-bible-salvation-is-by-faith-a...,"Mark 16:16 [1] says that ""Whoever believes and...","According to the Bible, salvation is by faith ..."
3,1.1.1.1,1.1.1,Con,according-to-the-bible-salvation-is-by-faith-a...,The portion of Mark where this verse appears i...,"According to the Bible, salvation is by faith ..."
4,1.1.1.2,1.1.1,Con,according-to-the-bible-salvation-is-by-faith-a...,This passage may imply that all who believe an...,"According to the Bible, salvation is by faith ..."


In [3]:
import re

def nettoyer(t):
    if not isinstance(t, str):      # protège contre les NaN après un rechargement CSV
        return ""
    t = re.sub(r'\[\d+\]', " ", t)         # 1. marqueurs [n]
    t = re.sub(r'http\S+', " ", t)         # 2. URL
    t = re.sub(r'\s+', " ", t).strip() # 3. espaces multiples puis strip
    # 4. (optionnel) espace avant la ponctuation
    return t

In [4]:
df["texte"] = df["content"].apply(nettoyer)

In [5]:
# Cellule 4 : vérifications avant/après
modifies = df["content"] != df["texte"]
print(modifies.sum())                           # part de textes modifiés
df.loc[modifies, ["content", "texte"]].sample(5)   # échantillon avant/après
print(df[df["texte"].str.split().str.len() < 3].shape[0])                           # nombre de textes de moins de 3 mots

47977
155


In [6]:
suivi = [('départ', len(df))]
suivi

[('départ', 97102)]

Résoudre les renvois

Trasnforme -> See 1.1.5. en vrai texte, pour récupérer des relations réelles et ne pas entraîner le modèle sur des phrases qui ne sont pas des arguments.

In [7]:
ref = re.compile(r"^->\s*See\s+([\d\.]+?)\.?$")
df["ref_id"] = df["texte"].str.extract(ref)[0]
df["is_ref"] = df["ref_id"].notna()
print(df["is_ref"].sum(), df["is_ref"].mean())                      # nombre et part de renvois

# dictionnaire (débat, id) -> texte, construit AVANT toute modification
lookup = df[~df["is_ref"]].set_index(["debate", "id"])["texte"].to_dict()

df.loc[df["is_ref"], "texte"] = [lookup.get((d, r)) for d, r in zip(df.loc[df["is_ref"], "debate"], df.loc[df["is_ref"], "ref_id"])]
# renvois dont la cible est introuvable ou elle-même un renvoi : texte = None
print(df["texte"].isna().sum())                      # combien
df = df.dropna(subset=["texte"])
suivi.append(("renvois résolus", len(df)))

5365 0.055251179172416634
0


Retirer les textes trop courts

Nous enlevons les textes vides ou sans contenu après nettoyage, qui n’apportent aucun signal.

In [8]:
df["n_mots"] = df["texte"].str.split().str.len()
print(df.loc[df["n_mots"] < 3, ["texte"]].sample(3))    # regarder avant de couper
df = df[df["n_mots"] >= 5]
suivi.append(("textes courts retirés", len(df)))

                         texte
89703      www.khanacademy.org
94527              Heat vision
94108  Neuroweapons (Category)


Langue

Nous selectionnons et ne concervons que l'anglais comme langue

In [9]:
DetectorFactory.seed = 0        # sinon le résultat peut varier d'une exécution à l'autre

In [10]:
def detecter(texte):
    try:
        meilleur = detect_langs(str(texte))[0]
        return meilleur.lang, meilleur.prob
    except LangDetectException:
        return "unknown", 0.0

resultats = df["content"].apply(detecter)
df['lang'] = resultats.str[0]
df['lang_prob'] = resultats.str[1]

In [11]:
res = df["texte"].apply(detecter)               # votre fonction
df["lang"], df["lang_prob"] = res.str[0], res.str[1]

# langue majoritaire par débat, sans compter les "unknown"
lang_debat = df[df["lang"] != "unknown"].groupby("debate")["lang"].agg(lambda s: s.mode()[0])
df["lang_debate"] = df["debate"].map(lang_debat)
print(pd.crosstab(df["debate"], df["lang_debate"]))                         # voir les débats mixtes
df = df[df["lang_debate"] == "en"]
suivi.append(("débats non anglais retirés", len(df)))

lang_debate                                         el   en  sk  sv
debate                                                             
according-to-the-bible-salvation-is-by-faith-al...   0  353   0   0
affirmative-action-useful-once-outdated-today-1715   0  781   0   0
all-humans-should-be-vegan-2762                      0  700   0   0
are-electric-guitars-better-than-acoustic-guita...   0  136   0   0
are-ghosts-real-23956                                0  269   0   0
...                                                 ..  ...  ..  ..
will-sex-robots-advance-sexual-liberation-5359       0  617   0   0
will-the-impact-of-ai-in-the-educational-sector...   0  134   0   0
will-we-ever-have-world-peace-41566                  0  191   0   0
would-the-world-be-a-better-place-without-human...   0  232   0   0
οι-έλληνες-πιλότοι-είναι-από-τους-καλύτερους-στ...  50    0   0   0

[238 rows x 4 columns]


Construction des paires positives

In [12]:
base = df[["debate", "id", "texte"]]
paires = df.merge(base, left_on=["debate", "parent"], right_on=["debate", "id"],
                suffixes=("", "_parent"))
paires = paires.rename(columns={"texte": "texte_enfant", "texte_parent": "texte_parent", "type": "label"})
paires = paires[["debate", "id", "parent", "texte_parent", "texte_enfant", "label"]]
suivi.append(("paires positives", len(paires)))

Traitement des doublons

In [13]:
print(f'Doublon : {df.duplicated(subset=["debate", "texte"]).sum()}')

Doublon : 5457


In [14]:
paires.groupby(["debate", "id"])["label"].nunique().gt(1).sum()   # paires contradictoires
paires = paires.drop_duplicates(subset=["debate", "id"])
suivi.append(("paires dédoublonnées", len(paires)))

In [15]:
import numpy as np

In [16]:
def cle(titre): 
    # fonction pour extraire une clé de regroupement à partir du titre du débat
    # par exemple, on peut prendre les 3 premiers mots du titre
    mots = titre.split()
    return ' '.join(mots[:3]) if len(mots) >= 3 else titre
paires["groupe"] = paires["debate"].apply(cle)      # ou une clé issue du titre

groupes = paires["groupe"].unique()
rng = np.random.default_rng(42)
groupes = rng.permutation(groupes)
n = len(groupes)
train_g, val_g, test_g = groupes[: int(0.7*n)], groupes[int(0.7*n): int(0.85*n)], groupes[int(0.85*n):]
paires["split"] = paires["groupe"].map({**{g: "train" for g in train_g}, **{g: "val" for g in val_g}, **{g: "test" for g in test_g}})

# Contrôles
print(paires["split"].value_counts())
print(paires.groupby("split")["label"].value_counts(normalize=True))
assert set(train_g).isdisjoint(test_g)

split
train    63941
val      17290
test     13804
Name: count, dtype: int64
split  label
test   Con      0.546436
       Pro      0.453564
train  Pro      0.506842
       Con      0.493158
val    Con      0.520243
       Pro      0.479757
Name: proportion, dtype: float64


Rengistrement des fichiers nettoyés

In [17]:
for nom in ["train", "val", "test"]:
    paires[paires["split"] == nom].to_csv(f"../cleanData/{nom}.csv", index=False)
pd.DataFrame(suivi, columns=["étape", "lignes"]).to_csv("../cleanData/suivi.csv", index=False)